# DE3: Real-time payments

Starter notebook for [DE3](../vignettes/de3_realtime_payments/README.md). Paste each step's prompt into the Cortex Code sidebar with this notebook open: Cortex Code adds the cells under the matching heading, and you run them.

Work in the **payments-hol** Workspace so Cortex Code can find the reference files in `vignettes/`.

## 1. Create the Postgres instance

**What this does:** creates **HOL_PAYMENTS_PG**, a Snowflake Postgres database that plays the part of the bank's payment engine, the operational system where payments are written as they happen. It runs inside Snowflake, so there's no server to set up. It takes 3 to 5 minutes. Postgres shows its admin password only once, so the helper saves it straight into a Snowflake secret instead of printing it.

Cells for this step: `create_pg` (Python).

## 2. Reveal the connection and connect

**What this does:** connects this notebook to the new Postgres database and creates the table the payment engine writes to, `payments.payment_events`. You copy the host, user and password from the Lab Companion into the cell below. The table starts empty. It has a primary key, which lets the mirror in the next step copy updates and deletes, not just new rows.

Cells for this step: `source_table` (Python).

In [ ]:
# Paste the values from the Lab Companion > DE3 > Reveal Postgres connection, then run this cell.
# Keep the quotes. Don't share the password.
PG_HOST = "<host_here>"
PG_USER = "<user_here>"
PG_PASS = "<pg_pass_here>"


## 3. Mirror Postgres into Snowflake

**What this does:** switches on **Data Mirroring**, which copies every change in the Postgres table into Snowflake continuously, with no pipeline to build or schedule. The copy lands as an open Iceberg table in the new database `PAYMENTS_MIRROR`. Alongside it you get `$live`, which is only seconds behind Postgres, and `$changes`, a feed of every insert, update and delete.

Cells for this step: `mirror` (Python).

## 4. Stream payments into Postgres

**What this does:** pretends to be the payment engine. For 5 minutes it writes about 20 payments a second into Postgres, around 6,000 in total. Halfway through, one corporate client's SEPA payments start failing with "Beneficiary bank unavailable": that's the incident the rest of the module finds.

Cells for this step: `stream` (Python).

## 5. Measure how fresh the data is

**What this does:** checks how far behind Postgres the Snowflake copy is. It counts payments per minute, then measures the gap between the newest payment and now, in `$live` (about 20 to 40 seconds) and in the main table (up to a minute, because changes are merged every minute). That gap is the end-to-end delay from the payment engine to Snowflake.

Cells for this step: `freshness` (SQL).

## 6. Serve the ops floor from the interactive warehouse

**What this does:** runs the queries an operations dashboard would refresh every few seconds, such as failed payments by scheme and the clients with the most failed value, on the **interactive warehouse** `HOL_INTERACTIVE_WH`. It reads the mirrored Iceberg table directly, with no copy. Interactive warehouses are built for many people, apps or agents asking quick questions at once, not for heavy one-off analysis, so the cell switches back to `BARCLAYS_WH` at the end.

Cells for this step: `ops_floor` (SQL).

## 7. Flag incidents automatically

**What this does:** creates a dynamic table, `DT_PAYMENT_INCIDENTS`, that looks at the last 15 minutes of failed payments and flags any client with 5 or more failures for the same scheme and reason. It refreshes itself every minute, so a new problem shows up within a minute of it starting in Postgres. You then draft the message the operations team would send to the client's relationship manager.

Cells for this step: `incidents` (SQL).